### training data

In [ ]:

import cv2
import os
from glob import glob

# مجلد البيانات
base_input_dir = r"C:\Users\ToP NeT\Desktop\nthu_ddd\training_dataset"
# المجلد اللي بدنا نحفظ فيه الفريمات
output_dir = r"C:\Users\ToP NeT\Desktop\nthu_ddd\nthu_ddd1"

# كل ملفات الفيديو
video_files = glob(os.path.join(base_input_dir,"*", "*.avi"))
#عدد الفيديوهات الكلي
print(f"number of videos: {len(video_files)}")

for video_path in video_files:
    path_parts = video_path.split("\\")  #  لازم backslash للويندوز
    person_id = path_parts[-3]   # رقم الشخص
    scenario = path_parts[-2]    # اسم السيناريو 
    video_filename = os.path.splitext(path_parts[-1])[0]   # اسم الفيديو بدون الامتداد



    # نجمع السيناريو + اسم الفيديو
    scenario_video_name = f"{scenario}_{video_filename}"

    #  ملف الليبل 
    label_txt_path = os.path.join(base_input_dir, person_id, scenario,
                                  f"{person_id}_{video_filename}_drowsiness.txt")

    #  قراءة الليبلات وتخزينها 
    with open(label_txt_path, "r") as f:
        labels = list(f.read().strip())

    # مكان تخزين الصور للفيديو الواحد 
    video_output_dir = os.path.join(output_dir, person_id, scenario_video_name)
    drowsy_dir = os.path.join(video_output_dir, "drowsy")
    awake_dir = os.path.join(video_output_dir, "awake")
    # final output files
    os.makedirs(drowsy_dir, exist_ok=True)
    os.makedirs(awake_dir, exist_ok=True)

    # قراءة الفيديو
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))#return number of frames of the video
    print(f"\n{scenario_video_name} | frames: {total_frames} | labels: {len(labels)}")

    # تحديد معدل استخراج الفريمات
    step = 15 if scenario in ["night_noglasses", "nightglasses"] else 30

    frame_num = 0
    while cap.isOpened():
        success, frame = cap.read()#read frame 
        if not success or frame_num >= len(labels):
            break

        # ناخذ فريم كل step
        if frame_num % step != 0:
            frame_num += 1
            continue

        # نحدد الكلاس
        class_id = int(labels[frame_num])
        # 1 >>>drowsy
        # 0 >>> awake
        class_name = "awake" if class_id == 0 else "drowsy"

        # تخزين الصورة بنمط تسمية مميز وبامتداد jpg
        frame_name = f"{person_id}_{scenario_video_name}_frame{frame_num:04d}.jpg"
        save_path = os.path.join(video_output_dir, class_name, frame_name)

        # حفظ الفريم
        cv2.imwrite(save_path, frame)
        frame_num += 1

    cap.release()

print("all frames have been extracted")


### Testing data

In [ ]:
import os
import cv2
# مجلد testing_dataset
base_input_dir = r"C:\Users\ToP NeT\Desktop\nthu_ddd\testing_dataset"
  
output_dir= r"C:\Users\ToP NeT\Desktop\nthu_ddd\nthu_ddd1"

# كل الفيديوهات
video_files = []
for person_id in os.listdir(base_input_dir):
    person_path = os.path.join(base_input_dir, person_id)
    if not os.path.isdir(person_path):
        continue

    for file in os.listdir(person_path):
        if file.endswith(".mp4"):   # الفيديوهات mp4
            video_files.append(os.path.join(person_path, file))

print(f"total videos found: {len(video_files)}")

# نبدأ نسحب الفريمات
for video_path in video_files:
    path_parts = video_path.split("\\")  # للويندوز
    person_id = path_parts[-2] # رقم الشخص 
    video_filename = os.path.splitext(path_parts[-1])[0]   # اسم الفيديو بدون .mp4

    # ملف الليبل 
    label_txt_path = os.path.join(base_input_dir, person_id, f"{video_filename}ing_drowsiness.txt")

    if not os.path.exists(label_txt_path):
        print(f"no labels file: {label_txt_path}")
        continue

    with open(label_txt_path, "r") as f:
        labels = list(f.read().strip())

    # مجلد الخرج (يحتوي على awake / drowsy)
    video_output_dir = os.path.join(output_dir, person_id, video_filename)
    drowsy_dir = os.path.join(video_output_dir, "drowsy")
    awake_dir = os.path.join(video_output_dir, "awake")
    os.makedirs(drowsy_dir, exist_ok=True)
    os.makedirs(awake_dir, exist_ok=True)

    # قراءة الفيديو
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f"\n{video_filename} | frames: {total_frames} | labels: {len(labels)}")

    step = 60  # ناخذ كل 60 فريم

    frame_num = 0
    while cap.isOpened():
        success, frame = cap.read()
        if not success or frame_num >= len(labels):
            break

        if frame_num % step == 0:  # ناخذ كل 30
            class_id = int(labels[frame_num])
            class_name = "awake" if class_id == 0 else "drowsy"

            frame_name = f"{video_filename}_frame{frame_num:04d}.jpg"
            save_path = os.path.join(video_output_dir, class_name, frame_name)

            cv2.imwrite(save_path, frame)

        frame_num += 1

    cap.release()

print(" all frames have been extracted")


total videos found: 25

014_glasses_mix | frames: 10999 | labels: 10997

014_nightglasses_mix | frames: 14665 | labels: 14663

014_nightnoglasses_mix | frames: 2805 | labels: 2803

014_noglasses_mix | frames: 7884 | labels: 7882

014_sunglasses_mix | frames: 13120 | labels: 13118

017_glasses_mix | frames: 3994 | labels: 3992

017_nightglasses_mix | frames: 8550 | labels: 8548

017_nightnoglasses_mix | frames: 14623 | labels: 14621

017_noglasses_mix | frames: 15933 | labels: 15931

017_sunglasses_mix | frames: 3265 | labels: 3263

022_glasses_mix | frames: 2552 | labels: 2550

022_nightglasses_mix | frames: 8202 | labels: 8200

022_nightnoglasses_mix | frames: 8617 | labels: 8615

022_noglasses_mix | frames: 16849 | labels: 16847

022_sunglasses_mix | frames: 2567 | labels: 2565

025_glasses_mix | frames: 10693 | labels: 10691

025_nightglasses_mix | frames: 16952 | labels: 16950

025_nightnoglasses_mix | frames: 21054 | labels: 21052

025_noglasses_mix | frames: 2963 | labels: 2961

